[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/50_fsdp_step_solution.ipynb)

# 🔴 Solution: FSDP Training Step

Reference solution for a simulated FSDP training step: all-gather → forward/backward → reduce-scatter → update.

$$\theta_i \leftarrow \theta_i - \eta \cdot \text{scatter}_i\!\left(\nabla_{\theta} \mathcal{L}(\text{gather}(\theta_0, \ldots, \theta_{N-1}))\right)$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

def fsdp_step(param_shards, grad_fn, world_size):
    # 1. All-gather
    full_param = torch.cat(param_shards)
    # 2. Compute gradient
    grad = grad_fn(full_param)
    # 3. Reduce-scatter
    grad_shards = list(grad.chunk(world_size))
    # 4. Update each shard (lr=0.01)
    new_shards = [
        param_shards[i] - 0.01 * grad_shards[i]
        for i in range(world_size)
    ]
    return new_shards

In [ ]:
# Verify: one FSDP step should match equivalent SGD on the full parameter
torch.manual_seed(0)
world_size = 4
shard_size = 8

# Initial shards
param_shards = [torch.randn(shard_size) for _ in range(world_size)]

# Simple quadratic loss gradient: grad = 2 * param
grad_fn = lambda p: 2.0 * p

# FSDP step
new_shards = fsdp_step(param_shards, grad_fn, world_size)
fsdp_result = torch.cat(new_shards)

# Reference: plain SGD on full param
full_param = torch.cat(param_shards)
ref_result = full_param - 0.01 * grad_fn(full_param)

print("FSDP result shape:", fsdp_result.shape)  # expect (32,)
print("Max diff vs SGD reference:", (fsdp_result - ref_result).abs().max().item())  # expect ~0

In [ ]:
from torch_judge import check
check("fsdp_step")